In [2]:
import os
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

print("All libraries imported successfully.")

All libraries imported successfully.


In [9]:
PROJECT_DIR = Path.cwd().parent
DATA_DIR = PROJECT_DIR / "data" / "DOLOS"
PROTOCOL_DIR = DATA_DIR / "Training_Protocols"

RESULTS_DIR = PROJECT_DIR / "results"
FIGURES_DIR = RESULTS_DIR / "figures"
METRICS_DIR = RESULTS_DIR / "metrics"
MODELS_DIR = PROJECT_DIR / "models"

for folder in [FIGURES_DIR, METRICS_DIR, MODELS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("DOLOS directory:", DATA_DIR)
print("Dataset exists:", DATA_DIR.exists())
print("Protocols exist:", PROTOCOL_DIR.exists())

Project directory: c:\Users\prate\OneDrive\Desktop\Lie Detector AI
DOLOS directory: c:\Users\prate\OneDrive\Desktop\Lie Detector AI\data\DOLOS
Dataset exists: True
Protocols exist: True


In [10]:
for file in sorted(DATA_DIR.rglob("*")):
    if file.is_file():
        print(file.relative_to(DATA_DIR))

Dolos.xlsx
dolos_timestamps.csv
dolos_timestamps.txt
train_dolos.xlsx
Training_Protocols\female.csv
Training_Protocols\long.csv
Training_Protocols\male.csv
Training_Protocols\short.csv
Training_Protocols\test_fold1.csv
Training_Protocols\test_fold2.csv
Training_Protocols\test_fold3.csv
Training_Protocols\train_fold1.csv
Training_Protocols\train_fold2.csv
Training_Protocols\train_fold3.csv


In [11]:
DOLOS_FILE = DATA_DIR / "Dolos.xlsx"
TRAIN_FILE = DATA_DIR / "train_dolos.xlsx"

raw_df = pd.read_excel(DOLOS_FILE, header=None)

print("Raw Excel dimensions:", raw_df.shape)
display(raw_df.iloc[:5, :15])

for i in range(min(5, len(raw_df))):
    print(f"\nRow {i}:")
    print(raw_df.iloc[i].tolist())

Raw Excel dimensions: (1681, 49)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,General Information,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Visual/Facial,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,General face​,NaN,NaN,Eyebrows​,NaN,Eyes​,NaN
2,Video URL,File name of the video clip,"Label ""truth"" or ""deception""",Start time in the episode,End time in the episode,Name of the game show,Participants name,Participants gender,Smile ​,Laughter ​,Scowl ​,Frowning​,Raising​,Frequent Blinking,Exaggerated Opening ​
3,https://www.youtube.com/watch?v=UCPIBQDI0QM&li...,AN_WILTY_EP15_truth1,truth,02:06:00,02:11:00,Would I Lie To You,Carol Voderman,Female,"[2:10,2:11]",0,0,0,0,0,0
4,https://www.youtube.com/watch?v=UCPIBQDI0QM&li...,AN_WILTY_EP15_truth2,truth,02:25:00,02:32:00,Would I Lie To You,Carol Voderman,Female,0,0,0,0,0,0,0



Row 0:
['General Information', nan, nan, nan, nan, nan, nan, nan, 'Visual/Facial', nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, 'Gaze', nan, nan, nan, 'Gesture', nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, 'Vocal', nan, nan, nan, nan]

Row 1:
[nan, nan, nan, nan, nan, nan, nan, nan, 'General face\u200b', nan, nan, 'Eyebrows\u200b', nan, 'Eyes\u200b', nan, nan, 'Mouth Openness\u200b', nan, 'Mouth Lips\u200b', nan, nan, nan, 'Chin', nan, nan, nan, nan, nan, 'Head Movements\u200b', nan, nan, nan, nan, nan, nan, 'Hand\xa0Movements\u200b', nan, nan, nan, nan, nan, 'Arm movement', nan, 'Shoulder movement', 'Fluency', nan, nan, 'Arousal', nan]

Row 2:
['Video URL', 'File name of the video clip ', 'Label "truth" or "deception"', 'Start time in the episode', 'End time in the episode', 'Name of the game show', 'Participants name', 'Participants gender', 'Smile\xa0\u200b', 'Laughter\xa0\u200b', 'Scowl\xa0\u200b', 'Frowning\u200b', 'Rai

In [12]:
dolos_df = pd.read_excel(DOLOS_FILE, header=2)
train_df = pd.read_excel(TRAIN_FILE, header=2)

print("DOLOS shape:", dolos_df.shape)
print("Training file shape:", train_df.shape)

print("\nDOLOS columns:")
for col in dolos_df.columns:
    print(repr(col))

display(dolos_df.head())

DOLOS shape: (1678, 49)
Training file shape: (1678, 44)

DOLOS columns:
'Video URL'
'File name of the video clip '
'Label "truth" or "deception"'
'Start time in the episode'
'End time in the episode'
'Name of the game show'
'Participants name'
'Participants gender'
'Smile\xa0\u200b'
'Laughter\xa0\u200b'
'Scowl\xa0\u200b'
'Frowning\u200b'
'Raising\u200b'
'Frequent Blinking'
'Exaggerated\xa0Opening\xa0\u200b'
'Exaggerated Closing'
'Exaggerated Opening\u200b'
'Closed (awkward pauses)'
'Corners up\xa0\u200b'
'Corners\xa0down\u200b'
'Protruded\xa0\u200b'
'Retracted'
'Move Upwards'
'Move Downwards'
'Towards\xa0interlocutor\u200b'
'Up\xa0\u200b'
'Down\xa0\u200b'
'Sideways\xa0\u200b'
'Nod'
'Move Forward\xa0\u200b'
'Move Backward\xa0\u200b'
'Tilt'
'Side-turn\xa0\u200b'
'Shake'
'Head up'
'Rubbing'
'Scratching'
'Touching head/body/etc'
'Covering face/mouth/etc'
'Crossing fingers'
'Palm up'
'Open'
'Folded'
'Shrugs'
'Non-ah-mmh speech disturbances'
'Word and phrase repetitions'
'Silent pauses'
'Lou

,Video URL,File name of the video clip,"Label ""truth"" or ""deception""",Start time in the episode,End time in the episode,Name of the game show,Participants name,Participants gender,Smile ​,Laughter ​,...,Crossing fingers,Palm up,Open,Folded,Shrugs,Non-ah-mmh speech disturbances,Word and phrase repetitions,Silent pauses,Loudness,Tension
0,https://www.youtube.com/watch?v=UCPIBQDI0QM&li...,AN_WILTY_EP15_truth1,truth,02:06:00,02:11:00,Would I Lie To You,Carol Voderman,Female,"[2:10,2:11]",0,...,0,"[2:10,2:11]",0,0,0,0,0,0,0,0
1,https://www.youtube.com/watch?v=UCPIBQDI0QM&li...,AN_WILTY_EP15_truth2,truth,02:25:00,02:32:00,Would I Lie To You,Carol Voderman,Female,0,0,...,"[2:25,2:32]",0,0,0,0,0,0,0,0,0
2,https://www.youtube.com/watch?v=UCPIBQDI0QM&li...,AN_WILTY_EP15_truth3,truth,05:50:00,05:55:00,Would I Lie To You,Larry Lamb,Male,0,0,...,0,0,0,0,0,0,0,0,0,0
3,https://www.youtube.com/watch?v=UCPIBQDI0QM&li...,AN_WILTY_EP15_lie4,deception,07:03:00,07:06:00,Would I Lie To You,Larry Lamb,Male,0,0,...,0,0,0,0,0,0,0,0,0,0
4,https://www.youtube.com/watch?v=UCPIBQDI0QM&li...,AN_WILTY_EP15_lie5,deception,07:32:00,07:36:00,Would I Lie To You,Larry Lamb,Male,0,0,...,0,"[7:32,7:36]",0,0,0,0,0,0,0,0


In [13]:
def clean_column_name(name):
    name = str(name).strip().lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    return name.strip("_")

dolos_df.columns = [
    clean_column_name(col) for col in dolos_df.columns
]

train_df.columns = [
    clean_column_name(col) for col in train_df.columns
]

print("Normalized columns:")
print(dolos_df.columns.tolist())

Normalized columns:
['video_url', 'file_name_of_the_video_clip', 'label_truth_or_deception', 'start_time_in_the_episode', 'end_time_in_the_episode', 'name_of_the_game_show', 'participants_name', 'participants_gender', 'smile', 'laughter', 'scowl', 'frowning', 'raising', 'frequent_blinking', 'exaggerated_opening', 'exaggerated_closing', 'exaggerated_opening', 'closed_awkward_pauses', 'corners_up', 'corners_down', 'protruded', 'retracted', 'move_upwards', 'move_downwards', 'towards_interlocutor', 'up', 'down', 'sideways', 'nod', 'move_forward', 'move_backward', 'tilt', 'side_turn', 'shake', 'head_up', 'rubbing', 'scratching', 'touching_head_body_etc', 'covering_face_mouth_etc', 'crossing_fingers', 'palm_up', 'open', 'folded', 'shrugs', 'non_ah_mmh_speech_disturbances', 'word_and_phrase_repetitions', 'silent_pauses', 'loudness', 'tension']


In [14]:
print("Potential label columns:")
print([c for c in dolos_df.columns if "label" in c or "truth" in c])

print("\nPotential video columns:")
print([c for c in dolos_df.columns if "video" in c or "file" in c])

print("\nPotential participant columns:")
print([c for c in dolos_df.columns if "participant" in c or "subject" in c])

display(dolos_df.head(10))

Potential label columns:
['label_truth_or_deception']

Potential video columns:
['video_url', 'file_name_of_the_video_clip']

Potential participant columns:
['participants_name', 'participants_gender']


,video_url,file_name_of_the_video_clip,label_truth_or_deception,start_time_in_the_episode,end_time_in_the_episode,name_of_the_game_show,participants_name,participants_gender,smile,laughter,...,crossing_fingers,palm_up,open,folded,shrugs,non_ah_mmh_speech_disturbances,word_and_phrase_repetitions,silent_pauses,loudness,tension
0,https://www.youtube.com/watch?v=UCPIBQDI0QM&li...,AN_WILTY_EP15_truth1,truth,02:06:00,02:11:00,Would I Lie To You,Carol Voderman,Female,"[2:10,2:11]",0,...,0,"[2:10,2:11]",0,0,0,0,0,0,0,0
1,https://www.youtube.com/watch?v=UCPIBQDI0QM&li...,AN_WILTY_EP15_truth2,truth,02:25:00,02:32:00,Would I Lie To You,Carol Voderman,Female,0,0,...,"[2:25,2:32]",0,0,0,0,0,0,0,0,0
2,https://www.youtube.com/watch?v=UCPIBQDI0QM&li...,AN_WILTY_EP15_truth3,truth,05:50:00,05:55:00,Would I Lie To You,Larry Lamb,Male,0,0,...,0,0,0,0,0,0,0,0,0,0
3,https://www.youtube.com/watch?v=UCPIBQDI0QM&li...,AN_WILTY_EP15_lie4,deception,07:03:00,07:06:00,Would I Lie To You,Larry Lamb,Male,0,0,...,0,0,0,0,0,0,0,0,0,0
4,https://www.youtube.com/watch?v=UCPIBQDI0QM&li...,AN_WILTY_EP15_lie5,deception,07:32:00,07:36:00,Would I Lie To You,Larry Lamb,Male,0,0,...,0,"[7:32,7:36]",0,0,0,0,0,0,0,0
5,https://www.youtube.com/watch?v=UCPIBQDI0QM&li...,AN_WILTY_EP15_truth6,truth,08:58:00,09:04:00,Would I Lie To You,Russel Howard,Male,0,0,...,0,0,0,0,0,0,0,0,0,0
6,https://www.youtube.com/watch?v=UCPIBQDI0QM&li...,AN_WILTY_EP15_truth7,truth,09:48:00,09:51:00,Would I Lie To You,Russel Howard,Male,0,0,...,0,0,0,0,0,0,0,0,0,0
7,https://www.youtube.com/watch?v=UCPIBQDI0QM&li...,AN_WILTY_EP15_truth8,truth,17:04:00,17:08:00,Would I Lie To You,Jo Brand,Female,0,0,...,0,0,0,0,0,0,"[17:04,17:06]","[17:06,17:07]",0,0
8,https://www.youtube.com/watch?v=UCPIBQDI0QM&li...,AN_WILTY_EP15_truth9,truth,17:44:00,17:47:00,Would I Lie To You,Jo Brand,Female,0,0,...,0,0,0,0,0,0,0,0,0,0
9,https://www.youtube.com/watch?v=UCPIBQDI0QM&li...,AN_WILTY_EP15_lie10,deception,18:17:00,18:20:00,Would I Lie To You,David Mitchell,Male,"[18:19,18:20]",0,...,0,0,0,0,0,0,0,0,0,0


In [15]:
protocol_files = sorted(PROTOCOL_DIR.rglob("*.csv"))

print("Protocol CSV files:", len(protocol_files))

for file in protocol_files:
    print(file.relative_to(PROTOCOL_DIR))

Protocol CSV files: 10
female.csv
long.csv
male.csv
short.csv
test_fold1.csv
test_fold2.csv
test_fold3.csv
train_fold1.csv
train_fold2.csv
train_fold3.csv


In [16]:
for file in protocol_files:
    print("\n" + "=" * 60)
    print("File:", file.name)

    protocol_df = pd.read_csv(file, header=None)

    print("Dimensions:", protocol_df.shape)
    display(protocol_df.head(5))


File: female.csv
Dimensions: (317, 3)


,0,1,2
0,AN_WILTY_EP15_truth1,truth,female
1,AN_WILTY_EP15_truth2,truth,female
2,AN_WILTY_EP15_truth8,truth,female
3,AN_WILTY_EP15_truth9,truth,female
4,AN_WILTY_EP16_truth26,truth,female



File: long.csv
Dimensions: (885, 3)


,0,1,2
0,AN_WILTY_EP15_truth1,truth,female
1,AN_WILTY_EP15_truth2,truth,female
2,AN_WILTY_EP15_truth3,truth,male
3,AN_WILTY_EP15_truth6,truth,male
4,AN_WILTY_EP15_lie14,deception,male



File: male.csv
Dimensions: (1245, 3)


...

,0,1,2
0,AN_WILTY_EP17_lie9,deception,male
1,AN_WILTY_EP22_lie12,deception,male
2,AN_WILTY_EP22_lie16,deception,male
3,LS_WILTY_EP9_lie20,deception,male
4,LS_WILTY_EP4_lie2,deception,male



File: train_fold2.csv
Dimensions: (1095, 3)


,0,1,2
0,YW_WILTY_EP50_lie5,deception,male
1,AN_WILTY_EP18_lie12,deception,male
2,SB_WILTY_EP41_lie15,deception,male
3,YW_WILTY_EP55_lie8,deception,male
4,AN_WILTY_EP25_lie1,deception,male



File: train_fold3.csv
Dimensions: (1100, 3)


,0,1,2
0,AN_WILTY_EP17_lie9,deception,male
1,AN_WILTY_EP22_lie12,deception,male
2,AN_WILTY_EP22_lie16,deception,male
3,LS_WILTY_EP9_lie20,deception,male
4,LS_WILTY_EP4_lie2,deception,male


In [17]:
print("Columns and data types:")
display(pd.DataFrame({
    "column": dolos_df.columns,
    "dtype": dolos_df.dtypes.astype(str),
    "missing": dolos_df.isna().sum().values,
    "unique_values": dolos_df.nunique(dropna=True).values
}).to_string(index=False))

Columns and data types:


'                        column  dtype  missing  unique_values\n                     video_url object        0             76\n   file_name_of_the_video_clip object        0           1677\n      label_truth_or_deception object        0              6\n     start_time_in_the_episode object        0           1048\n       end_time_in_the_episode object        0           1079\n         name_of_the_game_show object        0              2\n             participants_name object        0            206\n           participants_gender object        0              8\n                         smile object        0            243\n                      laughter object        0             85\n                         scowl object        0             30\n                      frowning object        0            186\n                       raising object        0            694\n             frequent_blinking object        0            276\n           exaggerated_opening object        0        

In [18]:
LABEL_COL = "label_truth_or_deception"
VIDEO_COL = "file_name_of_the_video_clip"
PARTICIPANT_COL = "participants_name"

print("Label distribution:")
display(dolos_df[LABEL_COL].value_counts(dropna=False))

print("Duplicate clip IDs:", dolos_df[VIDEO_COL].duplicated().sum())
print("Missing labels:", dolos_df[LABEL_COL].isna().sum())
print("Missing participant IDs:", dolos_df[PARTICIPANT_COL].isna().sum())

print("\nProtocol files:")
for file in protocol_files:
    protocol_df = pd.read_csv(file, header=None)
    print(file.name, protocol_df.shape)
    print(protocol_df.head(3).to_string(index=False, header=False))

Label distribution:


label_truth_or_deception
deception    783
truth        636
Truth        114
Deception     76
lie           42
truth         27
Name: count, dtype: int64

Duplicate clip IDs: 1
Missing labels: 0
Missing participant IDs: 0

Protocol files:
female.csv (317, 3)
AN_WILTY_EP15_truth1 truth female
AN_WILTY_EP15_truth2 truth female
AN_WILTY_EP15_truth8 truth female
long.csv (885, 3)
AN_WILTY_EP15_truth1 truth female
AN_WILTY_EP15_truth2 truth female
AN_WILTY_EP15_truth3 truth   male
male.csv (1245, 3)
AN_WILTY_EP15_truth3     truth male
  AN_WILTY_EP15_lie4 deception male
  AN_WILTY_EP15_lie5 deception male
short.csv (769, 3)
  AN_WILTY_EP15_lie4 deception male
  AN_WILTY_EP15_lie5 deception male
AN_WILTY_EP15_truth7     truth male
test_fold1.csv (549, 3)
 SB_WILTY_EP29_lie1 deception male
 YW_WILTY_EP45_lie5 deception male
LS_WILTY_EP10_lie10 deception male
test_fold2.csv (553, 3)
 AN_WILTY_EP17_lie9 deception male
AN_WILTY_EP22_lie12 deception male
AN_WILTY_EP22_lie16 deception male
test_fold3.csv (548, 3)
 YW_WILTY_EP50_lie5 deception male
AN_WILTY_EP18_lie12 deception male
SB_WILTY_EP41_lie15 deception male
train_fold1.csv (1099, 3)
 AN_WIL

In [20]:
import re
import numpy as np
import pandas as pd

metadata_cols = [
    "video_url",
    "file_name_of_the_video_clip",
    "label_truth_or_deception",
    "start_time_in_the_episode",
    "end_time_in_the_episode",
    "name_of_the_game_show",
    "participants_name",
    "participants_gender"
]

feature_cols = [c for c in dolos_df.columns if c not in metadata_cols]

def convert_annotation(value):
    if value is None:
        return np.nan

    if isinstance(value, (list, tuple, np.ndarray)):
        return float(len(value))

    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value) if np.isfinite(value) else np.nan

    if pd.isna(value):
        return np.nan

    text = str(value).strip()

    if text.lower() in {"", "nan", "none"}:
        return np.nan

    numeric = pd.to_numeric(text, errors="coerce")
    if pd.notna(numeric):
        return float(numeric)

    # Convert timestamp/event lists into counts.
    if text.startswith("[") and text.endswith("]"):
        contents = text[1:-1].strip()
        if not contents:
            return 0.0
        return float(len([x for x in contents.split(",") if x.strip()]))

    return np.nan

X = dolos_df[feature_cols].copy()

for col in feature_cols:
    X[col] = X[col].map(convert_annotation)

X = X.dropna(axis=1, how="all")

y = (
    dolos_df[LABEL_COL]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({"truth": 0, "deception": 1})
)

valid_rows = y.notna()

X = X.loc[valid_rows].reset_index(drop=True)
y = y.loc[valid_rows].astype(int).reset_index(drop=True)
groups = dolos_df.loc[valid_rows, PARTICIPANT_COL].reset_index(drop=True)

print("Feature matrix:", X.shape)
print("Labels:", y.value_counts().to_dict())
print("Usable features:")
print(X.columns.tolist())

Feature matrix: (1636, 43)
Labels: {1: 859, 0: 777}
Usable features:
['smile', 'laughter', 'scowl', 'frowning', 'raising', 'frequent_blinking', 'exaggerated_opening', 'exaggerated_opening', 'exaggerated_closing', 'exaggerated_opening', 'exaggerated_opening', 'closed_awkward_pauses', 'corners_up', 'corners_down', 'protruded', 'retracted', 'move_upwards', 'move_downwards', 'towards_interlocutor', 'up', 'down', 'sideways', 'nod', 'move_forward', 'move_backward', 'tilt', 'side_turn', 'shake', 'head_up', 'rubbing', 'scratching', 'touching_head_body_etc', 'covering_face_mouth_etc', 'crossing_fingers', 'palm_up', 'open', 'folded', 'shrugs', 'non_ah_mmh_speech_disturbances', 'word_and_phrase_repetitions', 'silent_pauses', 'loudness', 'tension']


In [21]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
import joblib

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train participants:", groups.iloc[train_idx].nunique())
print("Test participants:", groups.iloc[test_idx].nunique())
print("Train labels:", y_train.value_counts().to_dict())
print("Test labels:", y_test.value_counts().to_dict())

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced"
    ))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("\nBaseline results")
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Precision:", round(precision_score(y_test, y_pred, zero_division=0), 4))
print("Recall:", round(recall_score(y_test, y_pred, zero_division=0), 4))
print("F1-score:", round(f1_score(y_test, y_pred, zero_division=0), 4))

print("\nClassification report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["truth", "deception"],
    zero_division=0
))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

joblib.dump(model, MODELS_DIR / "dolos_random_forest_baseline.joblib")
print("Model saved.")

Train shape: (1234, 43)
Test shape: (402, 43)
Train participants: 160
Test participants: 40
Train labels: {1: 669, 0: 565}
Test labels: {0: 212, 1: 190}

Baseline results
Accuracy: 0.5448
Precision: 0.5135
Recall: 0.7
F1-score: 0.5924

Classification report:
              precision    recall  f1-score   support

       truth       0.60      0.41      0.48       212
   deception       0.51      0.70      0.59       190

    accuracy                           0.54       402
   macro avg       0.56      0.55      0.54       402
weighted avg       0.56      0.54      0.54       402

Confusion matrix:
[[ 86 126]
 [ 57 133]]
Model saved.


In [22]:
import json
from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

METRICS_DIR.mkdir(parents=True, exist_ok=True)

metrics = {
    "model": "Random Forest",
    "dataset": "DOLOS",
    "features": int(X_train.shape[1]),
    "train_samples": int(len(X_train)),
    "test_samples": int(len(X_test)),
    "accuracy": float(accuracy_score(y_test, y_pred)),
    "precision": float(precision_score(y_test, y_pred, zero_division=0)),
    "recall": float(recall_score(y_test, y_pred, zero_division=0)),
    "f1_score": float(f1_score(y_test, y_pred, zero_division=0)),
    "confusion_matrix": confusion_matrix(y_test, y_pred).tolist()
}

with open(METRICS_DIR / "baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

report = classification_report(
    y_test,
    y_pred,
    target_names=["truth", "deception"],
    output_dict=True,
    zero_division=0
)

pd.DataFrame(report).transpose().to_csv(
    METRICS_DIR / "classification_report.csv"
)

print("Evaluation results saved successfully.")
print(metrics)

Evaluation results saved successfully.
{'model': 'Random Forest', 'dataset': 'DOLOS', 'features': 43, 'train_samples': 1234, 'test_samples': 402, 'accuracy': 0.5447761194029851, 'precision': 0.5135135135135135, 'recall': 0.7, 'f1_score': 0.5924276169265034, 'confusion_matrix': [[86, 126], [57, 133]]}
